In [1]:
# Add the 'src' directory to the sys.path
import sys
sys.path.append('../../src')

In [2]:
import datetime
from elasticsearch import Elasticsearch
from literev.libs.parsing import process_search_query_elasticsearch

In [3]:
ES_JUDICIARY_INDEX_NAME = "judiciary"

In [4]:
es = Elasticsearch(
    [""],
    basic_auth=("", ""),
)

es.cluster.health()

ObjectApiResponse({'cluster_name': 'docker-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 9, 'active_shards': 9, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 3, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 75.0})

In [5]:
query = "divorce"
date_begin = datetime.date(2020, 1, 1)
date_end = datetime.date(2021, 1, 1)

es_query = process_search_query_elasticsearch(query, date_begin, date_end)

In [6]:
response = es.search(
    index=ES_JUDICIARY_INDEX_NAME,
    body=es_query,
)

total = response["hits"]["total"]["value"]
print('query:divorce (with date range) Results --->', total)

query:divorce (with date range) Results ---> 58


In [7]:
# Collector

from __future__ import annotations
import datetime
from elasticsearch import Elasticsearch
from django.conf import settings
from literev.libs.parsing import process_search_query_elasticsearch
from dataclasses import dataclass
from typing import cast


@dataclass
class MetaData:
    doc_id: str
    summary: str
    document_text: str
    decision_date: datetime.date
    result: str

class ElasticSearchCollector():
    """Implements all essential methods for collecting data from elasticsearch sources."""

    es: Elasticsearch
    ES_PAGE_SIZE: int = 1000
    ES_INDEX_NAME: str = "judiciary"

    def __init__(self) -> None:
        self.es = Elasticsearch(
            [settings.ES_HOSTNAME],
            basic_auth=(settings.ES_USERNAME, settings.ES_PASSWORD),
        )


    def collect_documents(self, search: str, date_begin: datetime.date, date_end: datetime.date) -> list[MetaData]:
        """Retrieve articles based on the provided search parameters."""

        es_query = process_search_query_elasticsearch(
            search_query=search,
            start_date=date_begin,
            end_date=date_end,
        )

        # get all documents from every page response from elasticsearch
        documents = self.get_all_documents_from_es_response(es_query)

        result = []

        for doc in documents:
            metadata = self.extract_document_metadata(doc)

            if metadata:
                result.append(metadata)
            else:
                self.log(
                    f"This document does not have document_text field: {doc}"
                )

        return result

    def get_all_documents_from_es_response(
        self,
        es_query: dict[str, int | list[str] | dict[str, str]],
    ) -> list[dict[str, str]]:
        """Get all articles from elasticsearch response."""

        es_query["size"] = self.ES_PAGE_SIZE

        response = self.es.search(
            index=self.ES_INDEX_NAME, body=es_query, scroll="2m"
        )

        scroll_id = response["_scroll_id"]
        hits = response["hits"]["hits"]

        documents = []

        # process the first page from elasticsearch
        documents += self._process_documents_from_es_response_page(hits)

        # then we process the rest of the pages if they exist
        # by passing the scroll_id to es.scroll
        while hits:
            response = self.es.scroll(scroll_id=scroll_id, scroll="2m")
            hits = response["hits"]["hits"]
            documents += self._process_documents_from_es_response_page(hits)

        return documents

    def _process_documents_from_es_response_page(
        self, hits: list[dict[str, dict[str, str]]]
    ) -> list[dict[str, str]]:
        """Process all articles from elasticsearch response page."""
        documents = []
        for es_hit in hits:
            # get article from elasticsearch hit _source key
            article = es_hit["_source"]
            if article:
                documents.append(article)
        return documents

    def extract_document_metadata(
        self, document: dict[str, str]
    ) -> MetaData | None:
        """Create Metadata from source article."""

        doc_id = document.get("id")
        summary = document.get("summary", "")
        document_text = document.get("document_text")
        date_decision = document.get("date_decision")
        result = document.get("result", "")
       
        if document_text:
            metadata = MetaData(
                doc_id=doc_id,
                summary=summary,
                document_text=document_text,
                decision_date=cast(datetime.date, date_decision),
                result=result,
            )

            return metadata

        return None

    def get_max_articles(
        self, search: str, begin: datetime.date, end: datetime.date
    ) -> int:
        """Counts total number of articles for a given query."""
        es_query = process_search_query_elasticsearch(
            search_query=search,
            start_date=begin,
            end_date=end,
        )

        response = self.es.count(
            index=self.ES_INDEX_NAME,
            body=es_query
            )
        
        return int(response["count"])

    def create_document_from_metadata(self, metadata: MetaData) -> None:
        """Create document from metadata."""
        pass



metadata = ElasticSearchCollector().collect_documents(query, date_begin, date_end)

In [8]:
len(metadata)

58

In [10]:
from pprint import pprint
pprint(metadata[0])

MetaData(doc_id='1588572672',
         summary='Le jugement de divorce des parents du recourant ne prévoit '
                 "pas de contribution d'entretien en faveur de l'intéressé "
                 "au-delà de sa majorité. C'est en ce sens qu'il convient "
                 "d'interpréter le mot « ultérieurement » figurant dans ledit "
                 "jugement. Le droit de l'intimé d'ignorer la situation "
                 'financière du père du recourant pour autant que ceux-ci '
                 "signent une convention d'entretien peut souffrir de rester "
                 "indécis. Le recourant n'a en effet jamais produit une telle "
                 "convention. Les échanges de courriels avec l'avocate de son "
                 "père sont insuffisants. L'intéressé n'a en outre pas "
                 'démontré que son père lui versait effectivement un certain '
                 'montant pour son entretien. Recours rejeté.',
         document_text='     république et\n'
       